# Task 1 (Advanced) – Random Forest Classifier with Hyperparameter Tuning

Random Forests are an ensemble of decision trees that vote on the final prediction. They're robust to overfitting and give us free feature-importance scores as a bonus.

**Dataset:** Iris  
**Goal:** Find the best `n_estimators`, `max_depth`, and `min_samples_split` via Grid Search, then evaluate and interpret the winning model.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 1. Load & Prepare Data

In [ ]:
iris = sns.load_dataset('iris')

X = iris.drop('species', axis=1)
y = iris['species']

# Hold out 20 % for the final, unbiased evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## 2. Hyperparameter Tuning via Grid Search

Instead of guessing parameter values, we let Grid Search try every combination and pick the winner using 5-fold cross-validation.

In [ ]:
param_grid = {
    'n_estimators'    : [50, 100, 200],   # how many trees to grow
    'max_depth'       : [None, 3, 5, 10], # None = grow until leaves are pure
    'min_samples_split': [2, 5]           # minimum samples needed to split a node
}

rf_base = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,          # 5-fold cross-validation
    n_jobs=-1,     # use all CPU cores
    verbose=1
)
grid_search.fit(X_train, y_train)

print("Best parameters found:", grid_search.best_params_)

## 3. Evaluate the Best Model

In [ ]:
best_rf = grid_search.best_estimator_

# Cross-validation on the full dataset gives a more reliable accuracy estimate
cv_scores = cross_val_score(best_rf, X, y, cv=5)
print(f"Cross-validation accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Final evaluation on the held-out test set
y_pred = best_rf.predict(X_test)
print(f"\nTest set accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nFull Classification Report:")
print(classification_report(y_test, y_pred))

## 4. Feature Importance

Random Forests tell us which features drove most of the decisions. Higher importance = the model leaned on that feature more.

In [ ]:
importance_df = pd.DataFrame({
    'Feature'   : X.columns,
    'Importance': best_rf.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
import seaborn as sns
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='magma')
plt.title('Feature Importance – Random Forest (Iris)')
plt.xlabel('Mean Decrease in Impurity')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 5. Confusion Matrix

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    confusion_matrix(y_test, y_pred),
    annot=True,
    fmt='d',
    xticklabels=best_rf.classes_,
    yticklabels=best_rf.classes_,
    cmap='Greens'
)
plt.title('Confusion Matrix – Random Forest')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()